In [4]:
import numpy as np
import pandas as pd
from statsmodels.api import OLS, add_constant
price_data = pd.read_csv("../data/prices.csv", index_col = 0, parse_dates = True).dropna()
price_data.head()

,AAPL,AMD,BAC,CVX,JPM,KO,MSFT,NVDA,PEP,XOM
Date,,,,,,,,,,
2023-01-03,123.096024,64.019997,30.974270,151.901291,124.928711,57.519154,233.452820,14.300685,162.498108,95.434189
2023-01-04,124.365662,64.660004,31.556595,150.286133,126.093643,57.491737,223.240829,14.734250,162.099548,95.711967
2023-01-05,123.046814,62.330002,31.491901,152.992584,126.065750,56.833855,216.624466,14.250736,160.405838,97.853439
2023-01-06,127.574188,63.959999,31.806168,154.145020,128.478058,57.930328,219.177444,14.844140,164.028809,99.036171
2023-01-09,128.095840,67.239998,31.325510,152.940186,127.947151,57.208481,221.311462,15.612370,162.425613,97.190392


In [6]:
# Build our backtest function,  returns sharpe ratio
def backtest_pair(price_data, t1, t2, window, entry_z, cost = 0.0005): # Day 17, added trading cost to function

    # Using Log prices 
    x = np.log(price_data[t1])
    y = np.log(price_data[t2])

    # Estimate beta
    X = add_constant(x)
    model = OLS(y,X).fit()
    beta = model.params[1]

    # Spread
    spread = y - beta* x

    # Z-Score
    mean = spread.rolling(window).mean()
    std = spread.rolling(window).std()
    zscore = (spread - mean)/ std

    # Computer trading signals
    position = pd.Series(0, index = spread.index)
    position[zscore > entry_z] = 1
    position[zscore < entry_z] = -1
    position = position.ffill().fillna(0)

    # Sharpe Ratio
    spread_ret = spread.diff().dropna()
    strategy_ret = position.shift(1) * spread_ret

    # Transaction cost
    trades = position.diff().abs()
    strategy_ret = strategy_ret - trades*cost

    sharpe = strategy_ret.mean() / strategy_ret.std() * np.sqrt(252)

    return sharpe

In [7]:
# Run a Grid Search

windows = [15,20,26,35,50]
thresholds = [1.5, 2.0, 2.5, 3.0]

results = []

for w in windows:
    for z in thresholds:
        sharpe = backtest_pair(price_data, "XOM", "CVX", w, z)
        results.append((w,z,sharpe))

results_df = pd.DataFrame(results, columns = ["Window", "Entry_Z", "Sharpe"])
results_df.sort_values(by = "Sharpe", ascending = False)
results_df

C:\Users\yogst\AppData\Local\Temp\ipykernel_21036\1272909480.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Local\Temp\ipykernel_21036\1272909480.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Local\Temp\ipykernel_21036\1272909480.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = model.params[1]
C:\Users\yogst\AppData\Lo

,Window,Entry_Z,Sharpe
0,15,1.5,-0.177110
1,15,2.0,-0.321834
2,15,2.5,-0.020310
3,15,3.0,-0.010920
4,20,1.5,-0.240500
5,20,2.0,-0.328660
6,20,2.5,-0.305003
7,20,3.0,-0.178259
8,26,1.5,-0.385513
9,26,2.0,-0.600391


### Interpretation
Even after testing different threshold and zscore values, found that all give a negative sharpe value, suggesting the XOM and CVX pair is not economically profitable, although the pair was cointegrated.